# Strategia Content-Based: similarità testuale basata sul contenuto degli articoli
In un sistema di Content-Based Filtering, l'assunzione di base è semplice: se un utente ha apprezzato certi articoli in passato, sarà probabilmente interessato ad articoli simili in futuro. Sfruttando la similarità testuale, il processo si divide in due fasi principali:
1. **Creazione del profilo utente**: rappresentiamo l'utente nello stesso "spazio vettoriale" degli articoli di news. Il metodo baseline più efficace consiste nel calcolare il vettore medio (centroide) degli embedding degli articoli con cui l'utente ha già interagito (la sua `History`).
2. **Calcolo della similarità (scoring e ranking)**: confrontiamo il profilo utente con gli articoli candidati calcolando la Cosine Similarity. Più l'angolo tra i due vettori è piccolo (valore che si avvicina a 1), più alta è la rilevanza stimata dell'articolo raccomandato.

## Indice
* [Generazione degli embedding](#embedding-generation)
* [Strategia Content-Based: similarità testuale](#content-based-strategy-text-based-similarity)
* [Pipeline di valutazione](#evaluation-pipeline)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
import math
from collections import Counter
from tqdm import tqdm
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dataset").exists() and (PROJECT_ROOT.parent / "dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

NEWS_PATH = str(PROJECT_ROOT / "dataset" / "MINDsmall_dev" / "news.tsv")
BEHAV_PATH = str(PROJECT_ROOT / "dataset" / "MINDsmall_dev" / "behaviors.tsv")
EMB_PATH = str(PROJECT_ROOT / "content_based_text" / "embeddings" / "news_embeddings.npy")

In [ ]:
# percorso di news.tsv
file_path = NEWS_PATH

# Stessi nomi di colonna definiti da Microsoft
columns = ['News_ID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL', 'Title_Entities', 'Abstract_Entities']

df_news = pd.read_csv(file_path, sep='\t', names=columns)  # sep='\t' perché è un file TSV

# I valori mancanti in 'Abstract' e 'Title' vengono sostituiti con stringhe vuote,
# per evitare problemi durante la generazione degli embedding.
df_news['Abstract'] = df_news['Abstract'].fillna('')
df_news['Title'] = df_news['Title'].fillna('')

# Nuova colonna 'full_text' usata per generare gli embedding.
# 'Title' e 'Abstract' sono separati da " - " per combinare i due campi in un'unica stringa.
df_news['full_text'] = df_news['Title'] + " - " + df_news['Abstract']

# Mostra le prime 5 righe
print(df_news[['News_ID', 'Category', 'full_text']].head())

## Generazione degli embedding
Per costruire un sistema di raccomandazione Content-Based robusto, dobbiamo prima tradurre le informazioni testuali degli articoli (titolo e abstract) in rappresentazioni matematiche: questo processo è noto come generazione degli embedding.

Invece di usare semplici frequenze di parole o output grezzi di BERT, utilizziamo Sentence-Transformers (nello specifico il modello `all-MiniLM-L6-v2`). Questo modello è stato scelto perché altamente ottimizzato, leggero ed estremamente veloce su CPU. Nonostante le dimensioni ridotte, è finemente addestrato per mappare frasi semanticamente simili in uno spazio vettoriale condiviso ad alta dimensionalità (384 dimensioni), rendendolo particolarmente adatto al successivo calcolo della cosine similarity senza compromettere le prestazioni.

Lo script seguente inizializza il modello, elabora il testo combinato (titolo + abstract) in batch efficienti e salva la matrice numerica risultante su disco per un accesso rapido in fase di valutazione.

In [ ]:
print("Download del modello...")
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = df_news['full_text'].tolist()

print("Generazione degli embedding...")
# Generazione degli embedding (batch_size elabora i dati a lotti, più efficiente su dataset grandi)
# model.encode() prende una lista di frasi e restituisce i relativi embedding come array NumPy.
embeddings = model.encode(texts, batch_size=64, show_progress_bar=True)

# Salva gli embedding
np.save(EMB_PATH, embeddings)
print(f"Embedding salvati in {EMB_PATH}")

## Strategia Content-Based: similarità testuale
In questa sezione implementiamo la logica principale del sistema di raccomandazione Content-Based. L'approccio si basa sul calcolo della cosine similarity tra le rappresentazioni matematiche (embedding) della cronologia di lettura dell'utente e degli articoli candidati disponibili.

### Preparazione dei dati e calcolo della popolarità
Per prima cosa carichiamo gli embedding precalcolati e il dataset dei comportamenti utente (behaviors). Calcoliamo inoltre la popolarità globale di ciascun articolo, necessaria in seguito per calcolare la metrica di Novelty.

In [ ]:
# Carica gli embedding
embeddings = np.load(EMB_PATH)

# Mapping tra News_ID e il relativo indice nella matrice di embedding
news_id_to_idx = {news_id: idx for idx, news_id in enumerate(df_news['News_ID'])}
idx_to_news_id = {idx: news_id for idx, news_id in enumerate(df_news['News_ID'])}

# Carica i dati di comportamento (dev set)
behaviors_path = BEHAV_PATH
behaviors_columns = ['Impression_ID', 'User_ID', 'Time', 'History', 'Impressions']
df_behaviors = pd.read_csv(behaviors_path, sep='\t', names=behaviors_columns)

# Gestisce le cronologie mancanti (alcuni utenti possono essere completamente nuovi)
df_behaviors['History'] = df_behaviors['History'].fillna('')

# Calcola la popolarità degli articoli per la metrica di Novelty
# Contiamo quante volte ogni articolo compare nelle cronologie degli utenti
all_history_items = ' '.join(df_behaviors['History'].tolist()).split()
item_counts = Counter(all_history_items)
total_interactions = len(all_history_items)

# Dizionario con la probabilità che un articolo venga letto (popolarità)
# Se un articolo non è mai stato letto, assegniamo una probabilità molto piccola per evitare errori di log(0)
item_popularity = {
    news_id: count / total_interactions 
    for news_id, count in item_counts.items()
}
min_popularity = 1e-6 # Fallback per articoli completamente nuovi

### Funzioni principali di raccomandazione
Queste funzioni si occupano della creazione del profilo utente (usando l'embedding medio degli articoli letti) e dell'estrazione delle top-K raccomandazioni tramite Cosine Similarity.

In [ ]:
def build_user_profile(history_news_ids, embeddings_matrix, mapping_dict):
    """
    Crea il vettore embedding medio di un utente a partire dagli articoli che ha letto.
    """
    # Trova gli indici degli articoli nella cronologia dell'utente
    clicked_indices = [mapping_dict[nid] for nid in history_news_ids if nid in mapping_dict]

    if not clicked_indices:
        return None # L'utente non ha una cronologia valida: non è possibile costruire un profilo.

    # Recupera gli embedding degli articoli cliccati
    clicked_embeddings = embeddings_matrix[clicked_indices]

    # Calcola il vettore medio degli embedding cliccati per rappresentare il profilo utente
    user_profile_vector = np.mean(clicked_embeddings, axis=0)

    # Ridimensiona il vettore del profilo utente per essere compatibile con cosine_similarity (1, embedding_dim)
    return user_profile_vector.reshape(1, -1)


def recommend_articles(user_profile, embeddings_matrix, mapping_dict, df, top_k=5):
    """
    Raccomanda articoli in base al profilo utente e agli embedding di tutti gli articoli.

    NOTA: questa funzione è dimostrativa (non è usata nel ciclo di valutazione
    della cella successiva) e cerca tra TUTTI gli articoli del dataset, senza
    escludere quelli già presenti nella cronologia dell'utente. Nella pipeline
    di valutazione questo problema non si pone, perché i candidati vengono
    presi dalle impression del DEV set, che per costruzione del dataset MIND
    non includono gli articoli già cliccati in precedenza dall'utente.
    """
    if user_profile is None:
        return "Impossibile raccomandare: cronologia utente vuota."

    # Cosine similarity tra il profilo utente e gli embedding di tutti gli articoli
    similarities = custom_cosine_similarity(user_profile, embeddings_matrix)

    # Ottiene gli indici dei top-K articoli più simili (ordinati in modo decrescente)
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # Recupera le informazioni degli articoli raccomandati
    recommendations = []
    for idx in top_indices:
        news_id = idx_to_news_id[idx]
        title = df.iloc[idx]['Title']
        score = similarities[idx]
        recommendations.append({'News_ID': news_id, 'Title': title, 'Similarity_Score': score})

    return pd.DataFrame(recommendations)

### Metriche di valutazione
Per valutare in modo completo la strategia di raccomandazione, definiamo funzioni dedicate per il ranking (MRR), l'accuratezza top-K (Precision@K, Recall@K) e le misure beyond-accuracy (Novelty@K). Nota: l'AUC utilizza l'implementazione ROC AUC di scikit-learn, mentre l'nDCG viene calcolato localmente con una funzione dedicata durante il ciclo di valutazione.

In [ ]:
def custom_cosine_similarity(user_profile, embeddings_matrix):
    # Appiattisce il profilo utente per garantire un vettore 1D.
    u = user_profile.flatten()

    dot_products = np.dot(embeddings_matrix, u)

    # numeratore
    norms_matrix = np.linalg.norm(embeddings_matrix, axis=1)
    norm_u = np.linalg.norm(u)

    # denominatore
    denom = norms_matrix * norm_u
    denom[denom == 0] = 1e-10

    return dot_products / denom

def custom_ndcg_score(y_true, y_score, k=5):
    """
    nDCG@K per etichette binarie.
    """
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    # Ordine degli item in base agli score predetti, decrescente
    order = np.argsort(y_score)[::-1]
    y_true_sorted = y_true[order][:k]

    # DCG
    discounts = np.log2(np.arange(2, len(y_true_sorted) + 2))
    dcg = np.sum(y_true_sorted / discounts)

    # DCG ideale (IDCG)
    ideal_sorted = np.sort(y_true)[::-1][:k]
    ideal_discounts = np.log2(np.arange(2, len(ideal_sorted) + 2))
    idcg = np.sum(ideal_sorted / ideal_discounts)

    if idcg == 0.0:
        return 0.0

    return dcg / idcg

def calculate_mrr(y_true, y_score):
    """Calcola il Mean Reciprocal Rank (MRR)."""
    # Ordina le etichette in base agli score predetti, decrescente
    order = np.argsort(y_score)[::-1]
    y_true_sorted = np.array(y_true)[order]

    # Trova l'indice del primo elemento rilevante (cliccato)
    for i, label in enumerate(y_true_sorted):
        if label == 1:
            return 1.0 / (i + 1)
    return 0.0

def calculate_precision_recall_at_k(y_true, y_score, k=5):
    """Calcola Precision@K e Recall@K."""
    order = np.argsort(y_score)[::-1]
    y_true_sorted = np.array(y_true)[order]

    top_k_labels = y_true_sorted[:k]

    hits = sum(top_k_labels)
    total_relevant = sum(y_true)

    precision_at_k = hits / k
    recall_at_k = hits / total_relevant if total_relevant > 0 else 0.0

    return precision_at_k, recall_at_k

def calculate_novelty_at_k(recommended_item_ids, k=5):
    """
    Calcola la Novelty@K.
    Definita come la self-information media: -log2(probabilità)
    """
    top_k_items = recommended_item_ids[:k]
    novelty_score = 0.0

    for item_id in top_k_items:
        prob = item_popularity.get(item_id, min_popularity)
        novelty_score += -math.log2(prob)

    return novelty_score / k if k > 0 else 0.0

## Pipeline di valutazione
Ora che le funzioni principali e le metriche sono definite, eseguiamo un ciclo di valutazione sul set di validazione (`MINDsmall_dev/behaviors.tsv`). In questa fase simuliamo il comportamento reale del sistema di raccomandazione. Per ogni sessione utente (impression) nel dataset, lo script:
1. costruisce dinamicamente il profilo utente in base alla sua cronologia di lettura fino a quel momento;
2. estrae gli articoli candidati mostrati in quella specifica sessione e le relative etichette (cliccato o ignorato);
3. calcola la Cosine Similarity tra il profilo utente e ciascun articolo candidato;
4. calcola e memorizza le metriche di prestazione (AUC, MRR, nDCG, Precision, Recall e Novelty) per quella sessione.

Nota: per consentire test rapidi, il codice seguente è impostato per valutare solo le prime 2.000 impression. Una volta verificato che tutto funzioni correttamente, è possibile rimuovere `.head(2000)` per valutare l'intero set di sviluppo.

In [ ]:
# Inizializza le liste per raccogliere le metriche su tutte le impression valutate
all_auc, all_mrr, all_ndcg5 = [], [], []
all_p5, all_r5, all_novelty5 = [], [], []

# Valutazione sull'intero set DEV
eval_df = df_behaviors

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Valutazione impression"):

    history_ids = str(row['History']).split()
    impressions = str(row['Impressions']).split()

    # Salta se l'utente non ha cronologia o non ha impression valide
    if not history_ids or not impressions:
        continue

    # Costruisce il profilo utente (usando la funzione definita in precedenza)
    user_profile = build_user_profile(history_ids, embeddings, news_id_to_idx)
    if user_profile is None:
        continue

    # Effettua il parsing delle impression: 'News_ID-Label' -> separa in liste
    cand_ids = [imp.split('-')[0] for imp in impressions]
    y_true = [int(imp.split('-')[1]) for imp in impressions]

    # Salta se non ci sono click positivi oppure sono tutti positivi in questa sessione
    # AUC e nDCG non sono calcolabili se è presente una sola classe
    if sum(y_true) == 0 or sum(y_true) == len(y_true):
        continue

    # Estrae gli embedding dei candidati (fallback a zero se l'articolo non è stato trovato)
    cand_embeddings = []
    valid_cand_ids = []

    for c_id in cand_ids:
        if c_id in news_id_to_idx:
            cand_embeddings.append(embeddings[news_id_to_idx[c_id]])
            valid_cand_ids.append(c_id)
        else:
            cand_embeddings.append(np.zeros(embeddings.shape[1]))
            valid_cand_ids.append(c_id)

    cand_embeddings = np.array(cand_embeddings)

    # Calcola gli score di Cosine Similarity
    y_score = custom_cosine_similarity(user_profile, cand_embeddings)

    # --- CALCOLO DELLE METRICHE ---

    # 1. AUC
    try:
        auc = roc_auc_score(y_true, y_score)
        all_auc.append(auc)
    except ValueError:
        pass # Gestisce i casi limite con una sola classe presente

    # 2. MRR
    mrr = calculate_mrr(y_true, y_score)
    all_mrr.append(mrr)

    # 3. nDCG@5
    ndcg5 = custom_ndcg_score(y_true, y_score, k=5)
    all_ndcg5.append(ndcg5)

    # 4. Precision@5 e Recall@5
    p5, r5 = calculate_precision_recall_at_k(y_true, y_score, k=5)
    all_p5.append(p5)
    all_r5.append(r5)

    # 5. Novelty@5
    # Ordina gli ID candidati per score decrescente per individuare i primi 5 raccomandati
    top_indices = np.argsort(y_score)[::-1]
    top_cand_ids = [valid_cand_ids[i] for i in top_indices]
    nov5 = calculate_novelty_at_k(top_cand_ids, k=5)
    all_novelty5.append(nov5)

# Calcola le medie finali
print("\n--- RISULTATI FINALI DI VALUTAZIONE ---")
print(f"AUC media:         {np.mean(all_auc):.4f}")
print(f"MRR media:         {np.mean(all_mrr):.4f}")
print(f"nDCG@5 media:      {np.mean(all_ndcg5):.4f}")
print(f"Precision@5:       {np.mean(all_p5):.4f}")
print(f"Recall@5:          {np.mean(all_r5):.4f}")
print(f"Novelty@5:         {np.mean(all_novelty5):.4f} (valori più alti indicano articoli più di nicchia/long-tail)")